# 05 — Model, bobot kelas, dan fitting akhir

**File:** `05_Model_dan_Bobot.ipynb`

Sekarang kita lihat bagaimana **satu kolom target aroma dipelajari oleh satu classifier**. Classifier adalah model untuk memberi skor kelas 0/1. Dalam Binary Relevance ada **109 classifier untuk setiap kandidat**, dan setiap classifier berisi banyak pohon boosting.

Di notebook ini kita membuat objek model **tanpa melatihnya**, menghitung bobot dari train, dan membaca daftar model final yang sudah ada. Inputnya adalah target train, konfigurasi, dan *recipe*: resep pengaturan model delapan kandidat.

Urutan belajar: **04_Grup_Split_dan_Fold.ipynb → notebook ini → 06_Optuna_CV_dan_OOF.ipynb**.

Kamu boleh membaca output yang tersimpan tanpa menjalankan sel. Kalau perlu menjalankan, pilih kernel `.venv` proyek, lalu **Restart Kernel and Run All**. Notebook memuat inputnya sendiri.

Source yang tampil dibaca langsung dari fungsi aktif di `KODE_ML/03_Kode/`. Menampilkan definisi `fit` tidak menjalankan training. Run All bawaan di sini hanya membaca hasil tersimpan dan memeriksa konfigurasi/bobot/file; gerbang training baru ada di notebook 07 dan default-nya mati.


## Persiapan: pastikan sumber berasal dari KODE_ML

Sel ini mencari folder `KODE_ML`, memuat kode dari sana, lalu menampilkan alamat **Source aktif**. Cek alamatnya supaya kamu tidak membaca salinan proyek lain.

`sys.dont_write_bytecode` mencegah Python membuat file cache saat belajar. Bagian ini menyiapkan lokasi kode, belum menyiapkan model terlatih.


In [1]:
from pathlib import Path
import sys

lokasi = Path.cwd().resolve()
kandidat = [p for induk in [lokasi, *lokasi.parents]
            for p in (induk, induk / "KODE_ML")]
kandidat.append(Path(r"C:\Users\ACER\Documents\Marvel\Skripsi\Project Skripsi\Perfume-MultiLabel-Classifier\KODE_ML"))
KODE_ML = next((p for p in kandidat if (p / "03_Kode/runtime.py").is_file()), None)
if KODE_ML is None:
    raise FileNotFoundError("Buka notebook dari folder KODE_ML yang lengkap.")
sys.dont_write_bytecode = True
sys.path.insert(0, str(KODE_ML.parent))
sys.path.insert(0, str(KODE_ML))
from IPython.display import display, Markdown
from belajar import tampilkan_sumber, tampilkan_awal
from research import runtime
assert Path(runtime.__file__).resolve().is_relative_to(KODE_ML.resolve())
print("Source aktif:", Path(runtime.__file__).resolve().parent)

Source aktif: C:\Users\ACER\Documents\Marvel\Skripsi\Project Skripsi\Perfume-MultiLabel-Classifier\KODE_ML\03_Kode


### Baca konfigurasi dan data train

`verify()` mengecek kesesuaian file, bentuk data, dan urutan baris. Lalu `load_partition("train")` mengambil fitur dan target pada bagian train yang sudah ditetapkan.

Daftar label dibaca dari manifest, yaitu catatan susunan data yang disimpan proyek. Sel ini tidak menulis daftar 109 label sendiri atau menggunakan label test untuk memilih parameter. Lihat output: jumlah baris berarti jumlah molekul, sedangkan jumlah label berarti banyak kolom target.


In [2]:
import numpy as np
import pandas as pd
from research import training, dataset, experiment
from research.results import main_run

verified = dataset.verify()
meta = verified[0]
train = dataset.load_partition("train", verified=verified)
labels = meta["labels"]
protocol = runtime.read(experiment.PROTOCOL)
run = main_run()
print({"baris_train": len(train["truth"]), "label": len(labels),
       "morgan": train["morgan"].shape, "deskriptor": train["descriptors"].shape})

{'baris_train': 5353, 'label': 109, 'morgan': (5353, 2048), 'deskriptor': (5353, 8)}


**Cara membaca bentuk datanya:** satu baris adalah satu molekul. Kolom pada target berisi label aroma, dan beberapa kolom boleh bernilai 1 pada baris yang sama.

A/C memakai **2048 bit Morgan**. B/D memakai **2048 bit Morgan + 8 deskriptor = 2056 fitur**. Angka 2048/2056 adalah jumlah fitur masukan, sedangkan 109 adalah jumlah target aroma. Target 0 berarti tidak tercatat, bukan bukti pasti aromanya tidak ada.


### Import dan konstanta `training.py`

Bagian awal `training.py` menunjukkan library dan pengaturan yang dipakai fungsi-fungsi berikut. XGBoost/LightGBM membangun model; `runtime` membantu mengatur penggunaan CPU, batas proses, serta penyimpanan kemajuan.

`Checksum` atau hash file dipakai untuk memeriksa apakah isi file berubah. Ini pemeriksaan file, berbeda fungsi dari fingerprint molekul.


In [3]:
tampilkan_awal(training)

**Import dan konstanta aktual:** `KODE_ML/03_Kode/training.py`. SHA-256: `4b8f34ca70a8263b3a8a25e7c0e2b3e46bc1e8767be2149ec7cf4c804024e184`.

```python
"""Satu classifier biner per label; bobot hanya dari bagian latih."""
import joblib
import numpy as np
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
import xgboost
from .runtime import compute_profile, Limits, read, write, digest, WEIGHTING_POLICY

```

### `training.make_model` — kode aktual

**File:** `KODE_ML/03_Kode/training.py`.

`make_model` menerima nama algoritma dan parameter, lalu membuat **objek model yang belum belajar**. Objek ini juga disebut *estimator*.

Yang perlu kamu ikuti di kode:
- `params` menyalin resep agar pengaturan milik pemanggil tidak ikut berubah.
- `compute_profile()` mengambil pengaturan CPU dan metode pohon XGBoost.
- `random_state=42` mengatur pengacakan model; ini berbeda dari seed pembagian fold.
- `**params` memasukkan resep baseline atau hasil Optuna. Pengaturan lain mengikuti default library dengan versi yang dikunci.

Belum ada `.fit()` di fungsi ini. Sel berikut hanya menampilkan kode lengkapnya, lokasi file, dan hash sumber.


In [4]:
tampilkan_sumber(training.make_model)

**Source aktual:** `KODE_ML/03_Kode/training.py`, baris 9–18. Kode di bawah dibaca dari objek yang diimpor. SHA-256: `4b8f34ca70a8263b3a8a25e7c0e2b3e46bc1e8767be2149ec7cf4c804024e184`.

```python
def make_model(algorithm, parameters):
    params = dict(parameters)
    profile = compute_profile()
    threads = profile["threads_per_model"]
    if algorithm == "xgb":
        return XGBClassifier(n_jobs=threads, random_state=42, eval_metric="logloss",
                             tree_method=profile["xgboost_tree_method"], **params)
    if algorithm == "lgbm":
        return LGBMClassifier(n_jobs=threads, random_state=42, verbosity=-1, **params)
    raise ValueError("Unknown algorithm.")

```

### Periksa delapan recipe aktual

Sel ini membaca resep delapan kandidat dari hasil utama, lalu membuat objek modelnya tanpa fitting.

Pada tabel, bandingkan jumlah **fitur**, tanda **tuned**, **seed_model**, dan parameter resep. Ini pengaturan yang benar-benar tersimpan, bukan contoh parameter baru.


In [5]:
rows = []
for key in experiment.CANDIDATES:
    recipe = runtime.read(run / key / "recipe.json")
    estimator = training.make_model(key.split("_")[0], recipe)
    rows.append({"kandidat": key, "fitur": 2048 + (8 if key[-1] in "BD" else 0),
                 "tuned": key in experiment.TUNED,
                 "seed_model": estimator.get_params()["random_state"], **recipe})
display(pd.DataFrame(rows).set_index("kandidat"))

,fitur,tuned,seed_model,n_estimators,learning_rate,max_depth,colsample_bytree,subsample,reg_alpha,reg_lambda,min_child_weight,gamma,min_child_samples,num_leaves,subsample_freq
kandidat,,,,,,,,,,,,,,,
xgb_A,2048,False,42,100,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
xgb_B,2056,False,42,100,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
xgb_C,2048,True,42,1200,0.005394,8.0,0.882710,0.684936,0.000019,0.008261,2.814509,2.623782,NaN,NaN,NaN
xgb_D,2056,True,42,1200,0.005394,8.0,0.882710,0.684936,0.000019,0.008261,2.814509,2.623782,NaN,NaN,NaN
lgbm_A,2048,False,42,100,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
lgbm_B,2056,False,42,100,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
lgbm_C,2048,True,42,400,0.028581,6.0,0.740236,0.875156,0.092028,5.274218,NaN,NaN,16.0,29.0,1.0
lgbm_D,2056,True,42,1100,0.007598,7.0,0.393844,0.896796,0.000038,0.267942,NaN,NaN,41.0,75.0,1.0


### `training.balanced_sample_weights` — kode aktual

**File:** `KODE_ML/03_Kode/training.py`.

`balanced_sample_weights` menerima target 0/1 untuk **satu label** pada bagian fitting. Hasilnya adalah bobot untuk setiap baris, supaya kelas yang lebih sedikit mendapat pengaruh lebih besar saat model belajar.

`counts` menghitung jumlah kelas 0 dan 1. Rumus `n / (2 * counts)` membagi jumlah baris `n` dengan dua kali jumlah anggota setiap kelas. `per_class[values]` lalu memberikan bobot yang sesuai ke tiap baris. Input kosong, bukan biner, atau hanya punya satu kelas akan ditolak.

Saat CV, bobot dihitung dari **bagian fitting fold itu saja**. Saat final fit, bobot dihitung dari seluruh train. Label pada validation/test tidak ikut menentukan bobot.


In [6]:
tampilkan_sumber(training.balanced_sample_weights)

**Source aktual:** `KODE_ML/03_Kode/training.py`, baris 21–30. Kode di bawah dibaca dari objek yang diimpor. SHA-256: `4b8f34ca70a8263b3a8a25e7c0e2b3e46bc1e8767be2149ec7cf4c804024e184`.

```python
def balanced_sample_weights(y):
    """Use the same fold-local weighting rule for XGBoost and LightGBM."""
    values = np.asarray(y, dtype=np.int8)
    if values.ndim != 1 or len(values) == 0 or not np.isin(values, [0, 1]).all():
        raise ValueError("Expected a non-empty binary target vector.")
    counts = np.bincount(values, minlength=2)
    if np.any(counts == 0):
        raise ValueError("Balanced weighting requires both classes.")
    per_class = len(values) / (2.0 * counts)
    return per_class[values]

```

### Buktikan rumus bobot pada label aktual

Kita memakai label pertama pada manifest sebagai contoh dari data asli. Perhatikan kolom `baris`, `bobot_per_baris`, dan `total_bobot`.

Kelas yang lebih sedikit mendapat bobot per baris lebih besar. **Total bobot kedua kelas harus sama**, meskipun jumlah barisnya berbeda. Tidak ada sampel yang digandakan atau target yang diganti.


In [7]:
y = train["truth"][:, 0]
weights = training.balanced_sample_weights(y)
table = pd.DataFrame([
    {"label": labels[0], "kelas": v, "baris": int((y == v).sum()),
     "bobot_per_baris": float(weights[y == v][0]), "total_bobot": float(weights[y == v].sum())}
    for v in (0, 1)
])
assert np.isclose(table.iloc[0].total_bobot, table.iloc[1].total_bobot)
display(table)

,label,kelas,baris,bobot_per_baris,total_bobot
0,acidic,0,5317,0.503385,2676.5
1,acidic,1,36,74.347222,2676.5


### `training.fit` — kode aktual

**File:** `KODE_ML/03_Kode/training.py`.

Inilah fungsi yang benar-benar melatih **satu classifier** jika dipanggil. Inputnya resep algoritma, fitur `x`, satu kolom target `y`, dan batas eksekusi; keluarannya model terlatih.

Urutannya: periksa batas → pastikan ada kelas 0 dan 1 → buat model → hitung bobot dari data fitting → jalankan `.fit()` → periksa batas lagi.

`Callback` adalah fungsi pemeriksa yang dipanggil saat fitting berlangsung. Di sini tugasnya mengecek batas proses/waktu, **bukan** menghentikan training berdasarkan skor validation (*early stopping*). Callback XGBoost dibersihkan setelah fitting agar tidak ikut disimpan.

Sel berikut hanya menampilkan fungsi. Pelatihan penuh dilakukan melalui program pengatur eksperimen (*runner*) di notebook 07.


In [8]:
tampilkan_sumber(training.fit)

**Source aktual:** `KODE_ML/03_Kode/training.py`, baris 33–53. Kode di bawah dibaca dari objek yang diimpor. SHA-256: `4b8f34ca70a8263b3a8a25e7c0e2b3e46bc1e8767be2149ec7cf4c804024e184`.

```python
def fit(algorithm, parameters, x, y, limits):
    limits.check(force=True)
    if np.unique(y).size != 2:
        raise ValueError("Training requires both classes.")
    model = make_model(algorithm, parameters)
    weights = balanced_sample_weights(y)
    if algorithm == "xgb":
        class Callback(xgboost.callback.TrainingCallback):
            def after_iteration(self, model, epoch, evals_log):
                limits.check()
                return False
        model.set_params(callbacks=[Callback()])
        model.fit(x, y, sample_weight=weights, verbose=False)
        model.set_params(callbacks=None)
    else:
        def callback(env):
            limits.check()
        callback.order = 5
        model.fit(x, y, sample_weight=weights, callbacks=[callback])
    limits.check(force=True)
    return model

```

### `training.fit_final_models` — kode aktual

**File:** `KODE_ML/03_Kode/training.py`.

`fit_final_models` mengatur pelatihan akhir satu classifier untuk setiap label, memakai seluruh data train dan resep yang sudah ditetapkan. Inputnya folder eksperimen (*run*), algoritma, resep, fitur/target train, dan urutan label.

Ikuti `enumerate(labels)`: nomor label menghubungkan nama aroma, kolom `y[:, j]`, dan file seperti `label_000.joblib`. `models.json` mencatat hubungan tersebut.

Jika file model sudah ada, hash-nya diperiksa sebelum dilewati. Model yang belum ada dilatih dan disimpan. Catatan kemajuan ini disebut *checkpoint*; tujuannya agar proses terputus bisa dilanjutkan tanpa mengulang semua classifier. Sel di bawah hanya menampilkan definisinya.


In [9]:
tampilkan_sumber(training.fit_final_models)

**Source aktual:** `KODE_ML/03_Kode/training.py`, baris 55–66. Kode di bawah dibaca dari objek yang diimpor. SHA-256: `4b8f34ca70a8263b3a8a25e7c0e2b3e46bc1e8767be2149ec7cf4c804024e184`.

```python
def fit_final_models(folder, algorithm, params, x, y, labels):
    saved = read(folder / "models.json") if (folder / "models.json").exists() else {}
    for j, name in enumerate(labels):
        if name in saved:
            if digest(folder / saved[name]["file"]) != saved[name]["sha256"]:
                raise ValueError("Final-model checkpoint changed.")
            continue
        model = fit(algorithm, params, x, y[:, j], Limits())
        filename = f"label_{j:03d}.joblib"
        joblib.dump(model, folder / filename)
        saved[name] = {"file": filename, "sha256": digest(folder / filename)}
        write(folder / "models.json", saved)

```

### Periksa model final yang sudah ada

Kita membaca `models.json` dan mengecek keberadaan file model, tanpa menjalankan `.fit()` atau menyimpan ulang model.

Jumlah **classifier_tersimpan** pada setiap kandidat harus sama dengan jumlah label: 109. Tabel contoh di bawahnya menunjukkan nama label dan file model yang terhubung dengannya.


In [10]:
rows = []
for key in experiment.CANDIDATES:
    models = runtime.read(run / key / "models.json")
    assert set(models) == set(labels)
    assert all((run / key / item["file"]).is_file() for item in models.values())
    rows.append({"kandidat": key, "classifier_tersimpan": len(models)})
display(pd.DataFrame(rows))
display(pd.DataFrame.from_dict(runtime.read(run / "lgbm_D/models.json"), orient="index").head())

,kandidat,classifier_tersimpan
0,xgb_A,109
1,xgb_B,109
2,xgb_C,109
3,xgb_D,109
4,lgbm_A,109
5,lgbm_B,109
6,lgbm_C,109
7,lgbm_D,109


,file,sha256
acidic,label_000.joblib,458271e32cb9e0229612cdeafec84a3fd732ac6434948a...
aldehydic,label_001.joblib,70eb2a22468a4b36eeef7d772e8c64d931c74e128f8649...
almond,label_002.joblib,42f5c5d5594f9a5b3cb83308abe829fc3ba6d79390f5cb...
amber,label_003.joblib,e0186a80498a8ee5c25055f0bbc393d4d26550b608e976...
animal like,label_004.joblib,610ca1d26b3221e007482ebb97bc9ff5fcffbfc00e016e...


## Bekal sidang

- **Kenapa ada 109 classifier?** Satu model biner mempelajari satu label. Untuk satu molekul, beberapa label bisa positif bersamaan.
- **Apakah `n_estimators` jumlah label?** Bukan. Itu jumlah iterasi boosting/pohon yang diminta dalam satu classifier.
- **Kenapa bobot memakai bagian fitting saja?** Agar proporsi kelas validation dan test tidak ikut menentukan cara model belajar.
- **Apa yang diperiksa notebook ini?** Data train, pengaturan model tanpa fit, bobot, dan daftar model tersimpan. Pemeriksaan ini tidak membuktikan bahwa training ulang akan memberi hasil identik atau lebih baik.

Lanjut ke **06_Optuna_CV_dan_OOF.ipynb** untuk tuning dan validasi.
